In [1]:
import sys

import polars as pl
import torch

from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.models.PatchMixer.common.configs import PatchMixerConfig
from modeling_module.models.PatchTST.common.configs import PatchTSTConfig, PatchTSTConfigWeekly
from modeling_module.utils.checkpoint import save_model_dict, load_model_dict

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

save_dir = DIR + 'fit/20251106_running'


True
1
12.8
2.9.0.dev20250716+cu128
NVIDIA GeForce RTX 5080
2.9.0.dev20250716+cu128


In [2]:
target_dyn_demand_weekly = pl.read_parquet(DIR + 'target_dyn_demand_weekly.parquet').sort(['oper_part_no', 'demand_dt'])
target_dyn_demand_weekly = (target_dyn_demand_weekly.group_by('oper_part_no', maintain_order = True).map_groups(lambda g: g.with_columns(pl.arange(1, len(g) + 1).alias('seq'))))

filtered_target = target_dyn_demand_weekly.group_by('oper_part_no').agg(pl.col('seq').max().alias('seq_max')).filter(pl.col('seq_max') > 260).select('oper_part_no')

target_dyn_demand_weekly = target_dyn_demand_weekly.join(filtered_target, on = 'oper_part_no', how = 'right').select(['oper_part_no', 'demand_dt', 'demand_qty'])
target_dyn_demand_weekly

oper_part_no,demand_dt,demand_qty
str,i64,f64
"""94KC-A0020""",201801,61.0
"""94KC-A0020""",201802,143.0
"""94KC-A0020""",201803,52.0
"""94KC-A0020""",201804,57.0
"""94KC-A0020""",201805,7.0
…,…,…
"""E5500-39013""",202652,5.0
"""E5500-39013""",202653,83.0
"""E5500-39013""",202701,32.0


In [3]:
plan_yyyymm = 201811
lookback = 52
horizon = 27
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

data_module = MultiPartDataModule(
    target_dyn_demand_weekly.sort(['oper_part_no', 'demand_dt']),
    lookback = lookback,
    horizon = horizon,
    batch_size = 64,
    val_ratio = 0.2,
    is_running = True
)

train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()

In [4]:
from modeling_module.training.model_trainers.total_train import run_total_train_weekly

mode_dict = run_total_train_weekly(
    train_loader,
    val_loader,
    lookback = lookback,
    horizon = horizon,
)

PatchMixer Base (Weekly)
[EXO-setup] inferred E=2, model.exo_dim=2, has_head=True

[train_patchmixer] ===== Stage 1/2 =====
  - spike: OFF
  - epochs: 10 | lr=0.0003 | horizon_decay=False
[train_patchmixer] Effective TrainingConfig:
{
  "device": "cuda",
  "lookback": 52,
  "horizon": 27,
  "epochs": 10,
  "lr": 0.0003,
  "weight_decay": 0.001,
  "t_max": 40,
  "patience": 100,
  "max_grad_norm": 30.0,
  "amp_device": "cuda",
  "loss_mode": "point",
  "point_loss": "huber",
  "huber_delta": 0.8,
  "q_star": 0.5,
  "use_cost_q_star": false,
  "Cu": 1.0,
  "Co": 1.0,
  "quantiles": [
    0.1,
    0.5,
    0.9
  ],
  "use_intermittent": true,
  "alpha_zero": 3.0,
  "alpha_pos": 1.0,
  "gamma_run": 0.3,
  "cap": null,
  "use_horizon_decay": false,
  "tau_h": 0.85,
  "val_use_weights": false,
  "spike_loss": {
    "enabled": false,
    "strategy": "mix",
    "huber_delta": 0.6,
    "asym_up_weight": 1.0,
    "asym_down_weight": 8.0,
    "mad_k": 1.5,
    "w_spike": 32.0,
    "w_norm": 1.0,


KeyboardInterrupt: 

In [ ]:
from modeling_module.models.Titan.common.configs import TitanConfig

pm_base_config = PatchMixerConfig(
        lookback = lookback,
        horizon = horizon,
        device = device,
        loss_mode = 'point',
        point_loss = 'mae'
    )

pm_quantile_config = PatchMixerConfig(
    lookback = lookback,
        horizon = horizon,
    device = device,
    loss_mode = 'quantile',
    quantiles = (0.1, 0.5, 0.9)
)

ti_config = TitanConfig(
            lookback=lookback,
            horizon=horizon,
            # 아래는 Titans.py에서 사용하는 공통 옵션들(필요 시 설정)
            input_dim=1,  # 데이터로더 입력 채널 수에 맞춰 조정
            d_model=256,
            n_layers=3,
            n_heads=4,
            d_ff=512,
            dropout=0.1,
            contextual_mem_size=256, persistent_mem_size=64,
            use_exogenous=True, exo_dim=2,  # 캘린더 sin/cos 자동 주입 조건
            final_clamp_nonneg=False,
        )
#
pt_config = PatchTSTConfig(
        device = device,
        loss_mode = 'auto',
        quantiles = (0.1, 0.5, 0.9)
    )

cfg_map = {
    "PatchMixer Base": pm_base_config,
    "PatchMixer Quantile": pm_quantile_config,
    # "Titan Base": ti_config,
    # "Titan LMM": ti_config,
    # "Titan Seq2Seq": ti_config,
    # "PatchTST Base": pt_config,
    # "PatchTST Quantile": pt_config
}

In [ ]:

models_only = {
    name: (pack["model"] if isinstance(pack, dict) and "model" in pack else pack)
    for name, pack in mode_dict.items()
}

# cfg_by_name도 맞춰서 준비(필요하면)
cfg_by_name = {
    name: (pack.get("cfg") if isinstance(pack, dict) else None)
    for name, pack in mode_dict.items()
}

builder_key_by_name = {
  "PatchMixer Base": "patchmixer_base",
  "PatchMixer Quantile": "patchmixer_quantile",
  # "Titan Base": "titan_base",
  # "Titan LMM": "titan_lmm",
  # "Titan Seq2Seq": "titan_seq2seq",
  # "PatchTST Base": "patchtst_base",
  # "PatchTST Quantile": "patchtst_quantile",
}
save_index = save_model_dict(
    models_only,
    save_dir,
    cfg_by_name=cfg_by_name,                 # None 가능
    builder_key_by_name=builder_key_by_name  # 기존 그대로
)

In [ ]:

# Load
from modeling_module.models.model_builder import (
    build_titan_base, build_titan_lmm, build_titan_seq2seq, build_patch_mixer_base, build_patch_mixer_quantile,
    build_patchTST_base, build_patchTST_quantile,
)

builders = {
    "patchmixer_base": lambda cfg: build_patch_mixer_base(cfg or PatchMixerConfig()),
    "patchmixer_quantile": lambda cfg: build_patch_mixer_quantile(cfg or PatchMixerConfig()),
    # "titan_base": lambda cfg: build_titan_base(ti_config or TitanConfig()),
    # "titan_lmm": lambda cfg: build_titan_lmm(ti_config or TitanConfig()),
    # "titan_seq2seq": lambda cfg: build_titan_seq2seq(ti_config or TitanConfig()),
    # "patchtst_base": lambda cfg: build_patchTST_base(cfg or PatchTSTConfigWeekly()),
    # "patchtst_quantile": lambda cfg: build_patchTST_quantile(cfg or PatchTSTConfigWeekly()),
}
loaded = load_model_dict(save_dir, builders, device = device)

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib, modeling_module.utils.plot_utils as pu
import modeling_module.training.forecaster as fo
importlib.reload(pu)
importlib.reload(fo)

def my_exo_cb(start_idx: int, Hm: int, device="cuda" if torch.cuda.is_available() else "cpu"):
    # exo_dim = 2 (sin, cos)
    return fo.make_calendar_exo(start_idx, Hm, period=52, device=device)

pu.plot_27w(
    models=loaded,           # {"PatchMixer": pm_model, "Titan": ti_model, ...}
    loader=val_loader,       # (xb, yb[, part_ids])
    device="cuda" if torch.cuda.is_available() else "cpu",
    mode="val",              # ← 검증 모드
    max_plots=5,
    out_dir=None,
    show=True,
    future_exo_cb=my_exo_cb
)